In [2]:
import json
import sys
import torch

print("Python :", sys.version)
print("Exécutable :", sys.executable)
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Python : 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Exécutable : C:\Users\Proprio\Desktop\GIF-4101\Projet\code\.venv\Scripts\python.exe
PyTorch : 2.14.1+cpu
CUDA disponible : False


In [9]:
import sys

!{sys.executable} -m pip install -U transformers accelerate safetensors


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import torch
import transformers
import accelerate
import safetensors

print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Accelerate :", accelerate.__version__)
print("Safetensors :", safetensors.__version__)

PyTorch : 2.14.1+cpu
Transformers : 5.20.2
Accelerate : 1.15.0
Safetensors : 0.8.0


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Chargement du tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Chargement du modèle...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu"
)

model.eval()

print("\nModèle chargé correctement.")
print("Appareil :", next(model.parameters()).device)
print("Nombre de paramètres :", sum(p.numel() for p in model.parameters()))

Chargement du tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Chargement du modèle...


Loading weights: 100%|██████████| 338/338 [00:10<00:00, 31.36it/s]



Modèle chargé correctement.
Appareil : cpu
Nombre de paramètres : 1543714304


In [12]:
messages = [
    {
        "role": "system",
        "content": (
            "Tu es un analyste spécialisé dans "
            "l'analyse des cadres éditoriaux."
        )
    },
    {
        "role": "user",
        "content": (
            "Analyse ce texte et identifie ses idées principales, "
            "ses valeurs et ses tensions intellectuelles : "
            "L'éducation doit permettre à chaque individu de "
            "développer son autonomie tout en participant "
            "activement à la vie collective."
        )
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

reponse = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(reponse)

Voici une analyse détaillée du texte que vous avez fourni :

1. Idées principales :
   - L'éducation est un outil crucial pour le développement personnel.
   - Elle doit favoriser l'autonomie individuelle.
   - Elle doit contribuer au bien-être social par la participation active.

2. Valeurs :
   - Autonomie personnelle (l'individu doit être libre d'exercer sa créativité).
   - Participation sociale (les individus doivent jouer un rôle actif dans leur communauté).

3. Tensions intellectuelles :
   - La contradiction entre individualisme et collectivisme.
   - Le compromis nécessaire entre les droits individuels et les responsabilités sociales.
   - La nécessité de trouver un équilibre entre l'apprentissage et la pratique réelle de ces compétences.

Ce texte souligne l'idée qu'une bonne éducation doit non seulement préparer les individus à vivre pleinement leurs vies personnelles mais aussi à participer constructivement à la société. Il met en avant deux principes clés : l'autonomie et 

In [37]:
def extractionDonnees(cheminFichier):
    with open(cheminFichier / "development.json", "r", encoding="utf-8") as fichier:
        development = json.load(fichier)
    articles = development["essays"]
    return articles

def nettoyer_html(html):
    soup = BeautifulSoup(html, "html.parser")

    # Remplacer les balises structurantes par des séparateurs
    for balise in soup.find_all([
        "p", "div", "br", "h1", "h2", "h3",
        "h4", "h5", "h6", "li", "blockquote"
    ]):
        balise.append("\n")

    texte = soup.get_text()

    # Nettoyer les espaces
    texte = re.sub(r"[ \t]+", " ", texte)
    texte = re.sub(r"\n\s*\n+", "\n\n", texte)

    return texte.strip()

def nettoyage(articles):
    for article in articles:
        article["clean_text"] = nettoyer_html(
            article.get("raw_text", "")
        )
    return articles

def validationArticles(articles):
    articles_valides = [
        a for a in articles
        if a.get("clean_text", "").strip()
    ]

    articles_exclus = [
        a for a in articles
        if not a.get("clean_text", "").strip()
    ]
    return articles_valides, articles_exclus

def evaluationDistributionLongueur(articles):

    longueurs = np.array([
        len(a["clean_text"].split())
        for a in articles
    ])

    print("Nombre d'articles :", len(longueurs))
    print("Minimum :", longueurs.min(), "mots")
    print("Maximum :", longueurs.max(), "mots")
    print("Moyenne :", round(longueurs.mean(), 1), "mots")
    print("Médiane :", int(np.median(longueurs)))
    print("Écart-type :", round(longueurs.std(), 1))

In [38]:
racine = Path.cwd().parent.parent  # fonctionne avec notebook
chemin = racine / "projet" / "resultats_essais"

articles = extractionDonnees(chemin)
articles = nettoyage(articles)
print(articles[0].keys())


articles_vides = [
    a for a in articles
    if not a.get("clean_text", "").strip()
]

print("Nombre d'articles concernés :", len(articles_vides))

articles_valides, articles_exclus = validationArticles(articles)
print("valide :", len(articles_valides), "exclus :",len(articles_exclus))

evaluationDistributionLongueur(articles)

dict_keys(['dossier_ids', 'essay_id', 'language_code', 'raw_text', 'source_author', 'source_category', 'source_published_at', 'source_updated_at', 'title', 'url', 'clean_text'])
Nombre d'articles concernés : 3
valide : 357 exclus : 3
Nombre d'articles : 360
Minimum : 0 mots
Maximum : 5642 mots
Moyenne : 869.7 mots
Médiane : 851
Écart-type : 565.6


## analyse token

In [39]:
import numpy as np

longueurs_tokens = []

for article in articles_valides:
    texte = article["clean_text"]

    nombre_tokens = len(
        tokenizer.encode(
            texte,
            add_special_tokens=False
        )
    )

    longueurs_tokens.append(nombre_tokens)

longueurs_tokens = np.array(longueurs_tokens)

print("Nombre d'articles :", len(longueurs_tokens))
print("Minimum :", longueurs_tokens.min())
print("Maximum :", longueurs_tokens.max())
print("Moyenne :", round(longueurs_tokens.mean(), 1))
print("Médiane :", int(np.median(longueurs_tokens)))
print("90e percentile :", int(np.percentile(longueurs_tokens, 90)))

print("\nArticles dépassant certains seuils :")

for seuil in [4000, 8000, 12000, 16000, 24000, 28000]:
    print(
        f"{seuil:5d} tokens :",
        np.sum(longueurs_tokens > seuil)
    )

Nombre d'articles : 357
Minimum : 178
Maximum : 11226
Moyenne : 1626.1
Médiane : 1609
90e percentile : 2710

Articles dépassant certains seuils :
 4000 tokens : 4
 8000 tokens : 2
12000 tokens : 0
16000 tokens : 0
24000 tokens : 0
28000 tokens : 0


In [69]:

prompt_systeme = """
Tu es un analyste spécialisé dans l'étude des caractéristiques
thématiques, intellectuelles et éditoriales des textes.

Tu dois analyser le texte intégral d'un article.

MÉTHODE D'ANALYSE :

1. Distingue rigoureusement les caractéristiques thématiques
   des caractéristiques éditoriales.

2. N'utilise aucune classification philosophique prédéfinie.
   Ne cherche pas à faire entrer le texte dans une école de pensée.

3. Identifie uniquement les caractéristiques réellement présentes
   dans le texte. Tu peux faire ressortir des caractéristiques
   inattendues.

4. Distingue les observations explicites des interprétations
   implicites.

5. Chaque caractéristique doit être accompagnée d'un extrait
   du texte qui justifie son identification.

6. Regroupe les formulations sémantiquement équivalentes afin
   d'éviter les répétitions.

7. N'invente aucune information. Si une dimension ne peut pas
   être déterminée, retourne une liste vide ou une chaîne vide
   selon le type du champ demandé.

FORMAT DE SORTIE OBLIGATOIRE :

- Réponds exclusivement avec un objet JSON valide.
- Le premier caractère de ta réponse doit être {.
- Le dernier caractère de ta réponse doit être }.
- N'utilise aucune balise Markdown.
- Ne produis jamais de bloc de code.
- N'ajoute aucune introduction, explication ou conclusion.
- Respecte exactement la structure JSON demandée.
- Utilise des guillemets doubles pour les clés et les chaînes.
- N'ajoute aucune virgule après le dernier élément d'une liste
  ou d'un objet.
"""


prompt_utilisateur = """
Analyse cet article et retourne un objet JSON ayant exactement
la structure suivante :

{{
    "thematique": {{
        "sujets": [],
        "domaines": [],
        "concepts": [],
        "problematique_centrale": ""
    }},
    "editorial": {{
        "idees_fondamentales": [],
        "valeurs": [],
        "principes": [],
        "vision_individu_societe": [],
        "rapport_au_changement": [],
        "approche_connaissance": [],
        "approche_argumentative": [],
        "posture_intellectuelle": [],
        "objectif_editorial": []
    }},
    "observations_libres": []
}}

Pour chaque caractéristique identifiée, utilise cette structure :

{{
    "caracteristique": "",
    "preuve": "",
    "nature": "explicite ou implicite"
}}

ARTICLE :

{texte}
"""

In [70]:
MAX_NEW_TOKENS = 1000 # permet de limiter la réponse

article_test = next(
    a for a in articles_valides
    if a["essay_id"] == "mysql_article_10024"
)
messages = [
    {
        "role": "system",
        "content": prompt_systeme
    },
    {
        "role": "user",
        "content": prompt_utilisateur.format(
            texte=article_test["clean_text"]
        )
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cpu")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False
    )

reponse = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(reponse)

{  
    "thematique": {
        "sujets": ["Institution d'enseignement supérieur", "Formation à distance"],
        "domaines": ["Politique de l'éducation", "Système d'enseignement supérieur"],
        "concepts": ["loi d'orientation", "formation à distance", "enseignement supérieur"],
        "problematique_centrale": "Développement et amélioration du système d'enseignement supérieur en Cameroun."
    },
    "editorial": {
        "idees_fondamentales": ["Réformes administratives", "Gouvernance éducative", "Partenariats internationaux"],
        "valeurs": ["Egalité", "Diversité", "Accès à l'éducation"],
        "principes": ["Conformité aux normes internationales", "Promotion de l'accessibilité", "Modernisation des systèmes d'enseignement"],
        "vision_individu_societe": ["Amélioration de l'accès à l'éducation", "Renforcement de la diversité académique", "Intégration des technologies dans l'éducation"],
        "rapport_au_changement": ["Adaptabilité face aux défis technologique

In [72]:



reponse_nettoyee = reponse.strip()

# Retirer les balises Markdown
if reponse_nettoyee.startswith("```json"):
    reponse_nettoyee = reponse_nettoyee[len("```json"):]

if reponse_nettoyee.endswith("```"):
    reponse_nettoyee = reponse_nettoyee[:-3]

# Convertir le JSON en dictionnaire Python
analyse = json.loads(reponse_nettoyee.strip())



Analyse correctement convertie !
<class 'dict'>


In [68]:
from sauvegardeEssais import *

descriptif = input("Description de cette série d'analyses : ")

dossier_resultats = racine / "projet" / "resultats_essais"

fichier_resultats = sauvegarder_analyses(
    analyse,
    dossier_resultats,
    descriptif
)

Fichier sauvegardé : C:\Users\Proprio\Desktop\GIF-4101\projet\resultats_essais\analyses_qwen_20261119_161601.json
Nombre d'analyses : 3


On calcule un embedding pour chaque caractéristique, puis on construit une représentation de l'article à partir de ses caractéristiques.

Avantage : plus riche, plus flexible et permet de comparer les articles selon différentes dimensions éditoriales.

Un article possédera maintenant plusieurs vecteurs. Il ne sera donc pas représenté par un seul point dans l'espace vectoriel, mais par un ensemble de caractéristiques.

C'est intéressant, car nous pourrons éventuellement comparer deux articles de plusieurs façons :
- leur proximité thématique ;
- leur proximité concernant les valeurs ;
- leur proximité concernant les principes ;
- leur proximité éditoriale globale.
Nous ne fusionnerons pas ces mesures immédiatement.

In [81]:
print(analyse)



{'thematique': {'sujets': ["Institution d'enseignement supérieur", 'Formation à distance'], 'domaines': ["Politique de l'éducation", "Système d'enseignement supérieur"], 'concepts': ["loi d'orientation", 'formation à distance', 'enseignement supérieur'], 'problematique_centrale': "Développement et amélioration du système d'enseignement supérieur en Cameroun."}, 'editorial': {'idees_fondamentales': ['Réformes administratives', 'Gouvernance éducative', 'Partenariats internationaux'], 'valeurs': ['Egalité', 'Diversité', "Accès à l'éducation"], 'principes': ['Conformité aux normes internationales', "Promotion de l'accessibilité", "Modernisation des systèmes d'enseignement"], 'vision_individu_societe': ["Amélioration de l'accès à l'éducation", 'Renforcement de la diversité académique', "Intégration des technologies dans l'éducation"], 'rapport_au_changement': ['Adaptabilité face aux défis technologiques', "Promotion de l'innovation académique"], 'approche_connaissance': ['Recherche approfon

In [76]:
analyses_qwen = []
analyses_qwen.append(analyse)

In [77]:

import pandas as pd

lignes = []

for analyse in analyses_qwen:

    essay_id = analyse["essay_id"]

    # Parcourir les deux espaces
    for espace in ["thematique", "editorial"]:

        for dimension, valeurs in analyse.get(espace, {}).items():

            # Transformer une valeur unique en liste
            if isinstance(valeurs, str):
                valeurs = [valeurs]

            for valeur in valeurs:

                # Cas où Qwen retourne un dictionnaire
                if isinstance(valeur, dict):
                    caracteristique = valeur.get("caracteristique", "")
                    preuve = valeur.get("preuve", "")
                    nature = valeur.get("nature", "")

                # Cas où Qwen retourne directement une chaîne
                else:
                    caracteristique = str(valeur)
                    preuve = ""
                    nature = ""

                if caracteristique.strip():
                    lignes.append({
                        "essay_id": essay_id,
                        "espace": espace,
                        "dimension": dimension,
                        "caracteristique": caracteristique,
                        "preuve": preuve,
                        "nature": nature
                    })

    # Ajouter les observations libres
    for observation in analyse.get("observations_libres", []):

        if isinstance(observation, dict):
            caracteristique = observation.get("caracteristique", "")
            preuve = observation.get("preuve", "")
            nature = observation.get("nature", "")
        else:
            caracteristique = str(observation)
            preuve = ""
            nature = ""

        if caracteristique.strip():
            lignes.append({
                "essay_id": essay_id,
                "espace": "editorial",
                "dimension": "observations_libres",
                "caracteristique": caracteristique,
                "preuve": preuve,
                "nature": nature
            })

# Construire le DataFrame
df_caracteristiques = pd.DataFrame(lignes)

df_caracteristiques.head(10)


KeyError: 'essay_id'